# TDE: Análise e Transformação de Dados com NumPy

**Contexto:** time de dados de uma rede de lojas. O notebook reúne as duas atividades práticas propostas, cada uma em três níveis de profundidade crescente.

Conteúdo abordado:
- Parte 1 (Análise de Dados no Varejo): operações vetorizadas, máscaras booleanas, `nan`, `np.where`, geradores aleatórios modernos, `np.unique`, `argmax`
- Parte 2 (Organização e Transformação de Dados): geração de sequências, tipagem, `reshape`, `vstack`, cópia de fatias, `ravel`, `argsort`, `newaxis`/broadcasting e resolução de sistemas lineares com `np.linalg.solve`

In [ ]:
import numpy as np

## Parte 1: Análise de Dados no Varejo

### Nível 1 (Básico): Faturamento Semanal e Filtros

**Passo 1:** histórico diário da semana em `pecas_vendidas`.
**Passo 2:** faturamento estimado, multiplicando o array por 50 (operação vetorizada, sem laço).
**Passo 3:** máscara booleana para os dias com mais de 200 peças vendidas.

In [ ]:
pecas_vendidas = np.array([150, 120, 90, 210, 300, 250, 180])

faturamento_diario = pecas_vendidas * 50
print('Faturamento diário:', faturamento_diario)

dias_de_pico = pecas_vendidas[pecas_vendidas > 200]
print('Dias de pico (>200 peças):', dias_de_pico)

### Nível 2 (Intermediário): Múltiplas Filiais e Falhas de Sistema

**Passo 1:** matriz 2D com as vendas de três filiais ao longo de quatro dias. A Loja A tem `np.nan` no 3º dia (falha de sistema).
**Passo 2:** total de peças por loja, somando pelas linhas (`axis=1`). Como a Loja A tem um `nan`, seu total também sai `nan`, o que já mostra o problema resolvido no próximo passo.
**Passo 3:** média diária geral da rede (uma média por dia, entre as três lojas) usando `np.nanmean`, que ignora os `nan` em vez de propagá-los.
**Passo 4:** matriz condicional com `np.where`, marcando 'Meta Atingida' (>= 200) ou 'Abaixo'. Como qualquer comparação com `nan` retorna `False`, a célula com falha de sistema cai automaticamente em 'Abaixo'.

In [ ]:
vendas_filiais = np.array([
    [200, 220, np.nan, 250],  # Loja A
    [150, 180, 160, 190],     # Loja B
    [300, 310, 290, 330],     # Loja C
])

total_por_loja = vendas_filiais.sum(axis=1)
print('Total por loja (com nan propagado):', total_por_loja)

media_diaria_rede = np.nanmean(vendas_filiais, axis=0)
print('Média diária da rede (ignorando nan):', media_diaria_rede)

status_metas = np.where(vendas_filiais >= 200, 'Meta Atingida', 'Abaixo')
print('Status da meta por loja/dia:')
print(status_metas)

### Nível 3 (Avançado): Categorização e Destaques de Marketing

**Passo 1:** amostra de 50 vendas sorteadas entre departamentos, usando `np.random.default_rng()` com semente fixa para resultados reprodutíveis.
**Passo 2:** volume de vendas por departamento com `np.unique(..., return_counts=True)`.
**Passo 3:** campanha campeã entre 5 campanhas simultâneas, usando o índice do maior valor (`argmax`).

In [ ]:
rng = np.random.default_rng(42)
departamentos = ['Eletrônicos', 'Roupas', 'Casa']
amostra_vendas = rng.choice(departamentos, size=50)

valores_unicos, contagens = np.unique(amostra_vendas, return_counts=True)
for depto, qtd in zip(valores_unicos, contagens):
    print(f'{depto}: {qtd} vendas')

faturamento_campanhas = np.array([12000, 45000, 23000, 89000, 31000])
campanha_campea = faturamento_campanhas.argmax()
print(f'Campanha campeã: índice {campanha_campea}, faturamento R$ {faturamento_campanhas[campanha_campea]}')

## Parte 2: Organização e Transformação de Dados

### Nível 1 (Básico): Geração de Sequências e Tipagem de Dados

**Passo 1:** 30 dias do mês com `np.arange(1, 31)`.
**Passo 2:** 5 metas de vendas igualmente espaçadas entre R$ 20.000 e R$ 30.000 com `np.linspace`.
**Passo 3:** estoque físico exportado como decimal, convertido para inteiro com `.astype(int)`.
**Passo 4:** últimos 5 dias do mês, invertidos (mais recente primeiro) com fatiamento de passo negativo.

In [ ]:
dias_do_mes = np.arange(1, 31)
print('Dias do mês:', dias_do_mes)

metas_semanais = np.linspace(20000, 30000, 5)
print('Metas de vendas:', metas_semanais)

estoque_decimal = np.array([10.5, 20.1, 30.9])
estoque_inteiro = estoque_decimal.astype(int)
print('Estoque convertido para inteiro:', estoque_inteiro)

ultimos_5_dias_invertidos = dias_do_mes[-5:][::-1]
print('Últimos 5 dias, do mais recente ao mais antigo:', ultimos_5_dias_invertidos)

### Nível 2 (Intermediário): Redimensionamento e Proteção de Dados

**Passo 1:** total de visitas do portal em 12 meses, reorganizado em 4 trimestres (4 linhas, 3 colunas) com `.reshape(4, 3)`.
**Passo 2:** visitas do primeiro semestre (marketing) e do segundo semestre (comercial) como matrizes separadas, combinadas de volta com `np.vstack`.
**Passo 3:** projeção a partir só do primeiro trimestre, usando `.copy()` no fatiamento para não alterar a matriz original por acidente.
**Passo 4:** matriz bidimensional achatada de volta para 1D com `.ravel()`, pronta para exportação em relatório corrido.

In [ ]:
visitas_12_meses = np.array([320, 350, 410, 390, 430, 470, 510, 495, 540, 560, 600, 630])

visitas_por_trimestre = visitas_12_meses.reshape(4, 3)
print('Visitas por trimestre:')
print(visitas_por_trimestre)

visitas_1_semestre = visitas_12_meses[:6].reshape(2, 3)
visitas_2_semestre = visitas_12_meses[6:].reshape(2, 3)
visitas_combinadas = np.vstack((visitas_1_semestre, visitas_2_semestre))
print('Visitas combinadas (1º + 2º semestre):')
print(visitas_combinadas)

projecao_1_trimestre = visitas_por_trimestre[0].copy()
projecao_1_trimestre[0] = 999  # alterar a cópia não afeta a matriz original
print('Projeção do 1º trimestre (cópia alterada):', projecao_1_trimestre)
print('Matriz original, intacta:')
print(visitas_por_trimestre)

relatorio_1d = visitas_combinadas.ravel()
print('Relatório achatado em 1D:', relatorio_1d)

### Nível 3 (Avançado): Ranking, Broadcasting e Resolução de Sistemas

**Passo 1:** notas de uma avaliação de pensamento computacional. `np.argsort()` retorna os índices da ordenação, revelando a posição de cada aluno no ranking.
**Passo 2:** preços base como vetor linha, transformado em coluna com `np.newaxis`, cruzado com múltiplos percentuais de desconto via broadcasting.
**Passo 3:** sistema linear para descobrir o custo unitário de placa robótica e sensor, resolvido com `np.linalg.solve`.

In [ ]:
notas = np.array([85, 92, 78, 95, 88])

indices_ranking = np.argsort(notas)
print('Índices em ordem crescente de nota:', indices_ranking)

indices_ranking_desc = np.argsort(notas)[::-1]
print('Ranking do melhor para o pior aluno (índices):', indices_ranking_desc)
for posicao, indice in enumerate(indices_ranking_desc, start=1):
    print(f'{posicao}º lugar: aluno {indice}, nota {notas[indice]}')

In [ ]:
precos_base = np.array([100, 150, 200])
precos_coluna = precos_base[:, np.newaxis]

percentuais_desconto = np.array([0.9, 0.8, 0.7])  # 10%, 20% e 30% de desconto

precos_com_desconto = precos_coluna * percentuais_desconto
print('Preços base como coluna:')
print(precos_coluna)
print('Matriz de preços com desconto (broadcasting):')
print(precos_com_desconto)

In [ ]:
# Projeto A: 2 placas + 1 sensor = R$ 500
# Projeto B: 1 placa - 1 sensor (estorno) = R$ 100
A = np.array([
    [2, 1],
    [1, -1],
])
b = np.array([500, 100])

custo_placa, custo_sensor = np.linalg.solve(A, b)
print(f'Custo unitário da placa robótica: R$ {custo_placa:.2f}')
print(f'Custo unitário do sensor: R$ {custo_sensor:.2f}')